# MERFISH — all 10 pairs
Latest main function with technology="merfish", pixels_per_100um=100. Reuse the historical cached slices/costs. Save matrices, decisions, four scores and historical-bound comparison. Failures stop execution.

In [ ]:
"""Run the packaged MERFISH automation and save one complete result per pair."""
from __future__ import annotations

import argparse
import json
import os
import sys
from pathlib import Path

HERE = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/cutoff_automation_test/merfish_automation')
ROOT = Path("/Users/salovjade/Library/CloudStorage/Dropbox/RAFTUP_after0320/raftup_reproduce")
SRC = Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src")
sys.path.insert(0, str(SRC))
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/raftup_cutoff_numba")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/raftup_cutoff_mpl")

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from raftup import _metrics_two_gpr
from raftup._cutoff_selection import run_automated_alignment

import hashlib, inspect
from raftup import _cutoff_selection as current_cs
source_root = Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src/raftup")
assert Path(current_cs.__file__).resolve() == (source_root / "_cutoff_selection.py").resolve(), "Restart kernel and import local repo"
assert inspect.signature(current_cs.run_automated_alignment).parameters["technology"].default == "dlpfc"
assert "pixels_per_100um" in inspect.signature(current_cs.run_automated_alignment).parameters, "Restart kernel: outdated main function"
SOURCE_FINGERPRINTS = {str(p.relative_to(source_root)): hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in sorted(source_root.rglob("*")) if p.is_file() and p.suffix in {".py", ".pkl"}}

PAIRS = {
    "0409": ("-0.04", "-0.09"), "0414": ("-0.04", "-0.14"),
    "0419": ("-0.04", "-0.19"), "0424": ("-0.04", "-0.24"),
    "0914": ("-0.09", "-0.14"), "0919": ("-0.09", "-0.19"),
    "0924": ("-0.09", "-0.24"), "1419": ("-0.14", "-0.19"),
    "1424": ("-0.14", "-0.24"), "1924": ("-0.19", "-0.24"),
}
OUT = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/merfish')


def calculate_scores(alignment, a, b):
    labels = np.concatenate([
        np.asarray(a.obs["original_clusters"]),
        np.asarray(b.obs["original_clusters"]),
    ])
    laa = _metrics_two_gpr.cal_layer_based_alignment_result_merfish_skill_all_zero(
        alignment, labels
    )
    if isinstance(laa, (tuple, list)):
        laa = laa[0]
    return {
        "LAA": float(laa),
        "GPR10": float(_metrics_two_gpr.GPR_original(
            alignment.copy(), a.obsm["spatial"], b.obsm["spatial"], 10, 1e-19
        )),
        "coverage_A": float(np.any(alignment != 0, axis=1).mean()),
        "coverage_B": float(np.any(alignment != 0, axis=0).mean()),
    }


def run(pair):
    pair_out = OUT / pair
    pair_out.mkdir(parents=True, exist_ok=True)
    a_id, b_id = PAIRS[pair]
    a = sc.read_h5ad(HERE / "slice_cache" / f"slice_{a_id}.h5ad")
    b = sc.read_h5ad(HERE / "slice_cache" / f"slice_{b_id}.h5ad")
    feature_cost = torch.load(
        ROOT / "0_preprocessing/feature_cost_in_MERFISH/cost_mac_reproducible"
        / f"{a_id}_{b_id}_somde_cost_matrix.pt",
        map_location="cpu", weights_only=False,
    )
    alignment, decision = run_automated_alignment(a, b, feature_cost, technology="merfish", pixels_per_100um=100)
    (pair_out / "source_fingerprints.json").write_text(json.dumps(SOURCE_FINGERPRINTS, indent=2))
    scores = calculate_scores(alignment, a, b)

    bounds = pd.read_csv(HERE / "historical_audit.csv", dtype={"pair": str})
    bounds["pair"] = bounds.pair.str.zfill(4)
    ref = bounds.loc[bounds.pair == pair].iloc[0]
    comparison = {
        "pair": pair,
        "coverage_A": scores["coverage_A"],
        "coverage_B": scores["coverage_B"],
        "coverage_min": min(scores["coverage_A"], scores["coverage_B"]),
        "rigid_A_to_B": float(ref.rigid_A_to_B),
        "rigid_B_to_A": float(ref.rigid_B_to_A),
        "rigid_min": float(ref.rigid_min),
        "count_upper_bound": float(ref.count_upper_bound),
        "A_above_directional_rigid": bool(scores["coverage_A"] >= ref.rigid_A_to_B),
        "B_above_directional_rigid": bool(scores["coverage_B"] >= ref.rigid_B_to_A),
        "min_above_rigid_min": bool(min(scores["coverage_A"], scores["coverage_B"]) >= ref.rigid_min),
        "A_below_count_upper": bool(scores["coverage_A"] <= ref.count_upper_bound),
        "B_below_count_upper": bool(scores["coverage_B"] <= ref.count_upper_bound),
    }
    np.save(pair_out / "alignment_matrix.npy", alignment)
    (pair_out / "cutoff_decision.json").write_text(json.dumps(decision, indent=2))
    (pair_out / "scores.json").write_text(json.dumps(scores, indent=2))
    (pair_out / "coverage_bounds.json").write_text(json.dumps(comparison, indent=2))
    print(json.dumps({"pair": pair, **decision, **scores, **comparison}), flush=True)




In [ ]:
SELECTED_PAIRS = list(PAIRS)
rows = []
for pair in SELECTED_PAIRS:
    run(pair)
    scores = json.loads((OUT / pair / "scores.json").read_text())
    rows.append(dict(pair=pair, **scores))
    pd.DataFrame(rows).to_csv(OUT / "all_pair_metrics.csv", index=False)
display(pd.DataFrame(rows))